# 2D Cylindrical Blast Wave Test

This notebook runs a simulation of a relativistic magnetized cylindrical blast wave with AsterX, and
visualizes the data saved in TSV and OpenPMD format.


### Test setup
In this test, an over-dense, over-pressured cylinder embedded in a uniformly magnetized, low-density
ambient medium drives a strong blast wave outward. Because the background magnetic field is
uniform, the magnetized fluid expands **anisotropically** — the blast is impeded across the field
lines and channeled along them — which makes this a demanding test of the relativistic MHD
evolution. The setup follows [Cipolletta et al., 2020](https://arxiv.org/pdf/1912.04794).

The fluid starts **at rest** ($v^i = 0$ everywhere). The density and pressure are constant in an
inner core, constant in the outer ambient medium, and joined by a smooth transition shell so the
initial data has no sharp discontinuity:

- $\rho(r),\, P(r) = \rho_{in},\, P_{in}; \ \ \ r < r_{in}$  
  $\\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ = \rho_{out},\, P_{out}; \ \ \ r > r_{out}$

- In the transition shell $r_{in} \le r \le r_{out}$, $\log\rho$ and $\log P$ vary **linearly** in
the cylindrical radius $r$:

$$
\rho(r) = \exp\!\left[\frac{(r_{out}-r)\,\log\rho_{in} +
(r-r_{in})\,\log\rho_{out}}{r_{out}-r_{in}}\right],
\qquad
P(r) = \exp\!\left[\frac{(r_{out}-r)\,\log P_{in} + (r-r_{in})\,\log P_{out}}{r_{out}-r_{in}}\right]
$$

- $r = \sqrt{x^2 + y^2}$ is the cylindrical radius; $r_{in}$ and $r_{out}$ are the inner and outer
radii of the transition shell.

- A **uniform** magnetic field is imposed via the vector potential $A_z = 0.1\,y$ (with $A_x = A_y =
0$), giving $B^i = (0.1,\, 0,\, 0)$.

- Parameter settings: $\rho_{in} = 10^{-2}$, $\rho_{out} = 10^{-4}$, $P_{in} = 1.0$, $P_{out} = 
3.0\times10^{-5}$, $r_{in} = 0.8$, $r_{out} = 1.0$, $B^x = 0.1$.

- Here, we use the ideal gas EOS with $\gamma = 4/3$.

- Initial data is set using the thorn ```AsterSeeds```. The code can be found in ```AsterX/AsterSeeds/src/2D_tests.cxx```.

- Grid domain is $[-6,+6] \times [-6,+6]$ with $50 \times 50$ cells.

- `Neumann` is used as boundary conditions in all directions.

In [ ]:
# this allows you to use "cd" in cells to change directories instead of requiring "%cd"
%automagic on
from etsetup import *

## 1. Steps to perform the simulation

First, let's first move to the Cactus folder:

In [ ]:
cd ~/Cactus

Now, let's create the parameter file to be used for this simulation:

In [ ]:
%%writefile ./par/Cylindrical_blast.par

ActiveThorns = "
    CarpetX
    IOUtil
    ODESolvers
    TimerReport
    ADMBaseX
    HydroBaseX
    TmunuBaseX
    AsterSeeds
    AsterX
    EOSX
    AsterMasks
"


# -------------------- Cactus --------------------------------------------------
Cactus::cctk_show_schedule = yes
Cactus::presync_mode       = "mixed-error"
CarpetX::poison_undefined_values = no

Cactus::terminate       = "time"
Cactus::cctk_final_time = 4.0



# -------------------- TimerReport ---------------------------------------------
TimerReport::out_every                  = 10
TimerReport::out_filename               = "TimerReport.asc"
TimerReport::output_all_timers_together = yes
TimerReport::output_all_timers_readable = yes
TimerReport::n_top_timers               = 50



# -------------------- CarpetX -------------------------------------------------
CarpetX::verbose = no

CarpetX::xmin = -6.0
CarpetX::ymin = -6.0
CarpetX::zmin = -0.32 # 2 cells

CarpetX::xmax = +6.0
CarpetX::ymax = +6.0
CarpetX::zmax = +0.32 # 2 cells

CarpetX::boundary_x = "neumann"
CarpetX::boundary_y = "neumann"
CarpetX::boundary_z = "neumann"

CarpetX::boundary_upper_x = "neumann"
CarpetX::boundary_upper_y = "neumann"
CarpetX::boundary_upper_z = "neumann"

CarpetX::ncells_x = 75
CarpetX::ncells_y = 75
CarpetX::ncells_z = 4

CarpetX::max_num_levels         = 1
CarpetX::regrid_every           = 100000
CarpetX::blocking_factor_x      = 1
CarpetX::blocking_factor_y      = 1
CarpetX::blocking_factor_z      = 1
CarpetX::regrid_error_threshold = 0.01

CarpetX::prolongation_type = "ddf"
CarpetX::ghost_size        = 2
CarpetX::dtfac             = 0.15



# -------------------- ODESolvers ----------------------------------------------
ODESolvers::method = "SSPRK3"



# -------------------- ADMBaseX -------------------------------------------------
ADMBaseX::initial_data    = "Cartesian Minkowski"
ADMBaseX::initial_lapse   = "one"
ADMBaseX::initial_shift   = "zero"
ADMBaseX::initial_dtlapse = "none"
ADMBaseX::initial_dtshift = "none"



# -------------------- AsterSeeds ----------------------------------------------
AsterSeeds::test_type = "2DTest"
AsterSeeds::test_case = "cylindrical blast"


# -------------------- AsterX --------------------------------------------------
AsterX::debug_mode = "no"
AsterX::flux_type = "HLLE"
AsterX::update_tmunu = "no"
AsterX::use_uct = "yes"
AsterX::vector_potential_gauge = "generalized Lorentz"
AsterX::lorenz_damp_fac = 46.875
AsterX::use_entropy_fix = "yes"

ReconX::reconstruction_method = "minmod"

Con2PrimFactory::c2p_prime = "Noble"
Con2PrimFactory::c2p_second = "Palenzuela"
Con2PrimFactory::c2p_tol = 1e-13
Con2PrimFactory::max_iter = 100
Con2PrimFactory::rho_abs_min = 1e-8
Con2PrimFactory::unit_test = "yes"
Con2PrimFactory::B_lim = 1e8
Con2PrimFactory::vw_lim = 1e8
Con2PrimFactory::Ye_lenient = "yes"

EOSX::evolution_eos = "IdealGas"
EOSX::gl_gamma = 1.3333333333
EOSX::poly_gamma = 1.3333333333
EOSX::rho_max = 1e8
EOSX::eps_max = 1e8

IO::out_dir = $parfile
IO::out_every = 10

CarpetX::out_tsv_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"
        
CarpetX::out_openpmd_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"

Then, submit the simulation using the following command:

In [ ]:
%%bash
rm -rf $HOME/simulations/Cylindrical_blast
./simfactory/bin/sim create-submit Cylindrical_blast \
  --parfile=./par/Cylindrical_blast.par --procs=2 --num-threads=1 --ppn-used=2 --walltime=00:20:00

The above command creates and runs the simulation ```Cylindrical_blast```, using the configuration ```sim```. The data is saved in the directory ```./simulations/Cylindrical_blast```.



In [ ]:
%%bash
# watch log output, following along as new output is produced
cd ~/Cactus
./simfactory/bin/sim show-output --follow Cylindrical_blast

## 2. Steps to visualize simulation data

The 2D data can be saved in both Silo format (which can be visualised, for instance, via VisIt) and in OpenPMD format. 

For further info on Silo, please visit: https://wci.llnl.gov/simulation/computer-codes/silo

For further info about OpenPMD, please visit:

- Official website:  https://www.openpmd.org
- GitHub repository: https://github.com/openPMD
- Documentation:     https://openpmd-api.readthedocs.io

Now, let's go back to the home directory:

In [ ]:
cd ~/

Import all the required modules:

In [ ]:
%pip install --user celluloid

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from celluloid import Camera
import openpmd_api as io

In [ ]:
%matplotlib inline

Open a .bp file (ADIOS2 extension) as an OpenPMD **'series'**, which is a collection of **'iterations'**, each of which contains **'records'**, which are sets of either structured data --- **'meshes'** --- or unstructured data --- **'particles'**. 

AsterX only outputs mesh data. Each record has one or more **'components'**: for example, a record representing a scalar field only has one component, while a record representing a vector field has three.

In [ ]:
home = os.environ["HOME"]
series = io.Series(os.path.join(home, "simulations",
                                "Cylindrical_blast",
                                "output-0000","Cylindrical_blast",
                                "Cylindrical_blast.it%08T.bp5"), io.Access.read_only)

All iterations in our series have the same structure, i.e., they contain
the same records, since they all represent the same output, just at
different times. 

Here we define an empty Python nested dictionary whose
structure, once full, will be:

Iteration 0:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

Iteration 1:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

[...]

In [ ]:
iter_rec_comp_dict = {}

Print info, register data chunks and fill the above dictionary:

In [ ]:
for index in series.iterations:
    iteration = str(index)

    print("\nIteration " + iteration + ":")
    print("==============")

    # Allocate an empty dictionary associated to this iteration
    iter_rec_comp_dict[iteration] = {}

    i = series.iterations[index]

    for key in i.meshes:
        print("Components of record \"" + key + "\":")

        # Allocate an empty dictionary associated to this record. Notice that
        # 'record' is an OpenPMD mesh object, so it's better to use 'key'
        # instead of 'record' as a key in the dictionary ('record' could also be
        # used, but it makes accessing the key clumsy).
        record = i.meshes[key]
        iter_rec_comp_dict[iteration][key] = {}

        # Load each component of each record as a 'data chunk', i.e., an
        # allocated, but STILL INVALID, NumPy array. Later we will flush all
        # chunks (i.e., basically, fill the NumPy arrays) at once: this leads
        # to better I/O performance compared to flushing a large number of
        # small chunks. That's why we bothered creating the nested dictionary:
        # in this way, we can access the valid NumPy arrays for plotting
        # without having to flush each single chunk.
        # *IMPORTANT*: DO NOT access data chunks until flushing has happened!
        for component in record:
            print("    > " + component)  # 'component' is a string
            iter_rec_comp_dict[iteration][key][component] = record[component].load_chunk()  # *INVALID* 3D NumPy array

        print("")

Flush all registered data chunks, which are now **VALID** 3D NumPy arrays:

In [ ]:
series.flush()

Visualize a 2D movie of the x-component of the magnetic field on the xy plane:

# Density

In [ ]:
import matplotlib.colors as colors

In [ ]:
# Select the desired record and component to plot
record    = "hydrobasex_rho_patch00_lev00" 
component = "hydrobasex_rho"

# Set up the axes for the plot and the colorbar
fig    = plt.figure(figsize = [4.8, 4])
axplot = fig.add_axes([0.12, 0.14, 0.72, 0.74])
axclb  = fig.add_axes([0.85, 0.14, 0.02, 0.74])
#axplot = fig.add_axes([0.12, 0.14, 0.7, 0.7])
#axclb  = fig.add_axes([0.8, 0.16, 0.05, 0.65])


# Set title and labels
axplot.set_title("$\\rho$", fontsize = 10., fontweight = "bold", color = "midnightblue")
axplot.set_xlabel("x", fontsize = 7.)
axplot.set_ylabel("y", fontsize = 7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))


# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D array containing the data
    array3D = iter_rec_comp_dict[iteration][record][component]
    
    # Plot on the (x, y) plane at the half-way value of z
    # Notice that the 3D array is stored in (z, y, x) order
    z_index = int(array3D.shape[0]/2)
    x0     = np.linspace(-6, 6, array3D.shape[2])
    y0     = np.linspace(-6, 6, array3D.shape[1])
    image   = axplot.pcolormesh(x0, y0, array3D[z_index, :, :],
                                cmap = "jet",
                                norm=colors.LogNorm(vmin=2e-5, vmax=2e-2))
    axplot.set_xlim(xmin=-5.9, xmax=5.9)
    axplot.set_ylim(ymin=-5.9, ymax=5.9)
    # Set up the colorbar
    axclb.tick_params(labelsize=7.0)
    fig.colorbar(image, cax = axclb, extend = "neither")
    
    # Print the current iteration
    axplot.text(3.0, 5.0, "Iteration " + iteration,
             fontsize = 8., fontweight = "bold", color = "white")

    # Take a snapshot of the figure at this iteration (needed later for the animation)
    camera.snap()

    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate()
HTML(animation.to_html5_video())

# Temperature

In [ ]:
# Select the desired record and component to plot
record_press    = "hydrobasex_press_patch00_lev00" 
component_press = "hydrobasex_press"

record_rho    = "hydrobasex_rho_patch00_lev00" 
component_rho = "hydrobasex_rho"

# Set up the axes for the plot and the colorbar
fig    = plt.figure(figsize = [4.8, 4])
axplot = fig.add_axes([0.12, 0.14, 0.72, 0.74])
axclb  = fig.add_axes([0.85, 0.14, 0.02, 0.74])
#axplot = fig.add_axes([0.12, 0.14, 0.7, 0.7])
#axclb  = fig.add_axes([0.8, 0.16, 0.05, 0.65])


# Set title and labels
axplot.set_title("$T$", fontsize = 10., fontweight = "bold", color = "midnightblue")
axplot.set_xlabel("x", fontsize = 7.)
axplot.set_ylabel("y", fontsize = 7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))


# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D array containing the data
    array3D_press = iter_rec_comp_dict[iteration][record_press][component_press]
    array3D_rho = iter_rec_comp_dict[iteration][record_rho][component_rho]
    
    # Plot on the (x, y) plane at the half-way value of z
    # Notice that the 3D array is stored in (z, y, x) order
    z_index = int(array3D.shape[0]/2)
    x0     = np.linspace(-6, 6, array3D.shape[2])
    y0     = np.linspace(-6, 6, array3D.shape[1])
    image   = axplot.pcolormesh(x0, y0, array3D_press[z_index, :, :]/array3D_rho[z_index, :, :],
                                cmap = "hot",
                                norm=colors.LogNorm(vmin=1e-1, vmax=100))
    axplot.set_xlim(xmin=-5.9, xmax=5.9)
    axplot.set_ylim(ymin=-5.9, ymax=5.9)
    # Set up the colorbar
    axclb.tick_params(labelsize=7.0)
    fig.colorbar(image, cax = axclb, extend = "neither")
    
    # Print the current iteration
    axplot.text(3.0, 5.0, "Iteration " + iteration,
             fontsize = 8., fontweight = "bold", color = "white")

    # Take a snapshot of the figure at this iteration (needed later for the animation)
    camera.snap()

    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate()
HTML(animation.to_html5_video())

# Magnetic Field

In [ ]:
# Select the desired record and component to plot
record_press    = "hydrobasex_press_patch00_lev00" 
component_press = "hydrobasex_press"

record_rho    = "hydrobasex_rho_patch00_lev00" 
component_rho = "hydrobasex_rho"

record_mag    = "hydrobasex_bvec_patch00_lev00"

component_magx = "hydrobasex_bvecx"
component_magy = "hydrobasex_bvecy"
component_magz = "hydrobasex_bvecz"

# Set up the axes for the plot and the colorbar
fig    = plt.figure(figsize = [4.8, 4])
axplot = fig.add_axes([0.12, 0.14, 0.72, 0.74])
axclb  = fig.add_axes([0.85, 0.14, 0.02, 0.74])

# Set title and labels
axplot.set_title("$|B|$", fontsize = 10., fontweight = "bold", color = "midnightblue")
axplot.set_xlabel("x", fontsize = 7.)
axplot.set_ylabel("y", fontsize = 7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))

# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D arrays containing the data
    array3D_bx    = iter_rec_comp_dict[iteration][record_mag][component_magx]
    array3D_by    = iter_rec_comp_dict[iteration][record_mag][component_magy]
    array3D_bz    = iter_rec_comp_dict[iteration][record_mag][component_magz]

    # Plot on the (x, y) plane at the half-way value of z
    # Notice that the 3D array is stored in (z, y, x) order
    z_index = int(array3D_press.shape[0]/2)

    norm_B = np.sqrt(array3D_bx[z_index, :, :]*array3D_bx[z_index, :, :]+
                    array3D_by[z_index, :, :]*array3D_by[z_index, :, :]+
                    array3D_bz[z_index, :, :]*array3D_bz[z_index, :, :])
    
    x0      = np.linspace(-6, 6, array3D_bx.shape[2])
    y0      = np.linspace(-6, 6, array3D_bx.shape[1])

    image   = axplot.pcolormesh(x0, y0, norm_B,
                                cmap = "viridis",
                                norm=colors.LogNorm(vmin=5e-3, vmax=1))

    # Overlay magnetic field lines on the same (x, y) slice
    Bx = array3D_bx[z_index, :, :]
    By = array3D_by[z_index, :, :]
    axplot.streamplot(x0, y0, Bx, By,
                      color="white", linewidth=0.6,
                      density=1.0, arrowsize=0.6)

    axplot.set_xlim(xmin=-5.9, xmax=5.9)
    axplot.set_ylim(ymin=-5.9, ymax=5.9)

    # Set up the colorbar
    axclb.tick_params(labelsize=7.0)
    fig.colorbar(image, cax = axclb, extend = "neither")

    # Print the current iteration
    axplot.text(3.0, 5.0, "Iteration " + iteration,
             fontsize = 8., fontweight = "bold", color = "white")

    # Take a snapshot of the figure at this iteration (needed later for the animation)
    camera.snap()
    
    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate()
HTML(animation.to_html5_video())

# Is the blast wave magnetically dominated?

In [ ]:
# Select the desired record and component to plot
record_press    = "hydrobasex_press_patch00_lev00"
component_press = "hydrobasex_press"

record_rho    = "hydrobasex_rho_patch00_lev00"
component_rho = "hydrobasex_rho"

record_mag    = "hydrobasex_bvec_patch00_lev00"

component_magx = "hydrobasex_bvecx"
component_magy = "hydrobasex_bvecy"
component_magz = "hydrobasex_bvecz"

# --- figure size: change these two numbers to resize the whole figure ---
# The panel layout below uses figure-fraction coordinates, so it rescales
# automatically -- you only ever need to touch fig_width / fig_height.
fig_width  = 12.0
fig_height = 4.0

# Set up the figure: 3 panels side by side, each the same relative width and
# height. We reuse the original single-panel fractional layout within each
# 1/3-wide slot.
fig = plt.figure(figsize=[fig_width, fig_height])

def panel_axes(i):
    axp = fig.add_axes([(i + 0.12) / 3, 0.14, 0.72 / 3, 0.74])
    axc = fig.add_axes([(i + 0.85) / 3, 0.14, 0.02 / 3, 0.74])
    return axp, axc

axplot1, axclb1 = panel_axes(0)
axplot2, axclb2 = panel_axes(1)
axplot3, axclb3 = panel_axes(2)

# Titles and labels (titles are static, set once)
for ax, ttl in [(axplot1, r"$|B|$"),
                (axplot2, r"$|B|^2/\rho$"),
                (axplot3, r"$|B|^2/2p$")]:
    ax.set_title(ttl, fontsize=10., fontweight="bold", color="midnightblue")
    ax.set_xlabel("x", fontsize=7.)
    ax.set_ylabel("y", fontsize=7.)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
    ax.yaxis.set_major_locator(plt.MaxNLocator(5))

# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D arrays containing the data
    array3D_press = iter_rec_comp_dict[iteration][record_press][component_press]
    array3D_rho   = iter_rec_comp_dict[iteration][record_rho][component_rho]
    array3D_bx    = iter_rec_comp_dict[iteration][record_mag][component_magx]
    array3D_by    = iter_rec_comp_dict[iteration][record_mag][component_magy]
    array3D_bz    = iter_rec_comp_dict[iteration][record_mag][component_magz]

    # Plot on the (x, y) plane at the half-way value of z
    # Notice that the 3D array is stored in (z, y, x) order
    z_index = int(array3D_bx.shape[0] / 2)

    Bx = array3D_bx[z_index, :, :]
    By = array3D_by[z_index, :, :]
    Bz = array3D_bz[z_index, :, :]
    pr = array3D_press[z_index, :, :]
    rh = array3D_rho[z_index, :, :]

    B2    = Bx * Bx + By * By + Bz * Bz   # |B|^2
    normB = np.sqrt(B2)                   # |B|

    x0 = np.linspace(-6, 6, array3D_bx.shape[2])
    y0 = np.linspace(-6, 6, array3D_bx.shape[1])

    # --- the three fields to plot (colormaps/norms are placeholders) ---
    panel_data = [
        (axplot1, axclb1, normB,        "viridis", colors.LogNorm(vmin=5e-3, vmax=1)),
        (axplot2, axclb2, B2 / rh,      "magma",  colors.LogNorm(vmin=1e-1, vmax=1000)),
        (axplot3, axclb3, B2 / (2 * pr), "plasma", colors.LogNorm(vmin=1e-2, vmax=1000)),
    ]

    for axp, axc, data, cmap, norm in panel_data:
        image = axp.pcolormesh(x0, y0, data, cmap=cmap, norm=norm)

        # Overlay magnetic field lines on the same (x, y) slice
        axp.streamplot(x0, y0, Bx, By,
                       color="black", linewidth=0.6,
                       density=1.0, arrowsize=0.6)

        axp.set_xlim(xmin=-5.9, xmax=5.9)
        axp.set_ylim(ymin=-5.9, ymax=5.9)

        axc.tick_params(labelsize=7.0)
        fig.colorbar(image, cax=axc, extend="neither")

    # Print the current iteration (once) — forced on top, with a backing box
    axplot1.text(0.03, 0.95, "Iteration " + iteration,
                   transform=axplot1.transAxes,        # axes-fraction coords, never off-screen
                   ha="left", va="top",
                   fontsize=10., fontweight="bold", color="black",
                   bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
                   zorder=10,                           # above pcolormesh + streamlines
                   clip_on=False)

    # Take a snapshot of all three panels at once (keeps them in sync)
    camera.snap()

    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate(interval=120, blit=False)
HTML(animation.to_html5_video())

You may delete the simulation directory via the following:


In [ ]:
%cd ~/
%rm -r ./simulations/Cylindrical_blast